In [ ]:
!pip install langchain transformers pypdf faiss-cpu sentence-transformers
!pip install langchain_community
!pip install langchain_huggingface

In [ ]:
from langchain.document_loaders import PyPDFLoader
from langchain.text_splitter import RecursiveCharacterTextSplitter
from langchain.embeddings import HuggingFaceEmbeddings
from langchain.vectorstores import FAISS

# Load and process the PDF
def build_and_save_faiss_store(pdf_path, store_path="faiss_store"):
    # Load the PDF into LangChain documents
    loader = PyPDFLoader(pdf_path)
    documents = loader.load()

    # Split documents into smaller chunks
    text_splitter = RecursiveCharacterTextSplitter(chunk_size=1000, chunk_overlap=200)
    split_docs = text_splitter.split_documents(documents)

    # Create FAISS vector store
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_store = FAISS.from_documents(split_docs, embeddings)

    # Save the store
    vector_store.save_local(store_path)
    print(f"FAISS store saved to {store_path}!")

# Example: Build and save FAISS store
pdf_path = "example.pdf"  # Replace with your PDF file
build_and_save_faiss_store(pdf_path)


In [ ]:
from langchain.vectorstores import FAISS
from transformers import GPT2LMHeadModel, GPT2Tokenizer

# Load the FAISS store
def load_faiss_store(store_path="faiss_store"):
    embeddings = HuggingFaceEmbeddings(model_name="sentence-transformers/all-MiniLM-L6-v2")
    vector_store = FAISS.load_local(store_path, embeddings,allow_dangerous_deserialization=True)
    return vector_store

# Initialize GPT-2 for generation
def setup_gpt2_pipeline():
    model_name = "gpt2"
    tokenizer = GPT2Tokenizer.from_pretrained(model_name)
    model = GPT2LMHeadModel.from_pretrained(model_name)
    return tokenizer, model

# Example: Load the store and GPT-2
vector_store = load_faiss_store()
tokenizer, gpt2_model = setup_gpt2_pipeline()


In [ ]:
import torch

def rag_with_gpt2(query, vector_store, tokenizer, gpt2_model):
    # Step 1: Retrieve relevant chunks
    retriever = vector_store.as_retriever()
    relevant_docs = retriever.get_relevant_documents(query)
    context = " ".join([doc.page_content for doc in relevant_docs])

    # Step 2: Prepare input for GPT-2
    input_text = f"Context: {context}\n\nQuestion: {query}\n\nAnswer:"
    inputs = tokenizer.encode(input_text, return_tensors="pt", truncation=True, max_length=1024)

    # Set attention mask
    attention_mask = torch.ones(inputs.shape, dtype=torch.long)


    outputs = gpt2_model.generate(
        inputs,
        attention_mask=attention_mask,
        max_new_tokens=200,
        num_beams=3,
        early_stopping=True,
        pad_token_id=tokenizer.eos_token_id
    )
    answer = tokenizer.decode(outputs[0], skip_special_tokens=True)

    return answer

# Example: Query with RAG
query = "What is the document about?"
response = rag_with_gpt2(query, vector_store, tokenizer, gpt2_model)
print("\n--- Generated Response ---")
print(response)

In [ ]:
# Get all documents from the vector store
documents = []
for i in range(len(vector_store.index_to_docstore_id)):
    doc_id = vector_store.index_to_docstore_id[i]
    doc = vector_store.docstore.search(doc_id)
    documents.append(doc)

# Print summary info
print(f"Total documents: {len(documents)}")
print("\nSample of documents:")
for i, doc in enumerate(documents[:3]):  # Show first 3 docs
    print(f"\nDocument {i+1}:")
    print(f"ID: {doc.id}")
    print(f"Metadata: {doc.metadata}")
    print("Content preview (first 200 chars):")
    print(doc.page_content[:200])
    print("-" * 80)

In [ ]:
# Get the actual vectors from FAISS index
vectors = vector_store.index.reconstruct_n(0, vector_store.index.ntotal)

print(f"Number of vectors: {len(vectors)}")
print(f"Vector dimension: {vectors[0].shape}")

# Look at first vector
print("\nFirst vector (first 10 dimensions):")
print(vectors[0][:10])

# Basic vector stats
print("\nVector statistics:")
print(f"Mean: {vectors.mean()}")
print(f"Min: {vectors.min()}")
print(f"Max: {vectors.max()}")

In [ ]:
# Perform similarity search
search_query = "What is the role of Indigenous communities in clean energy projects?"
similar_docs = vector_store.similarity_search(
    search_query,
    k=3  # get top 3 most similar chunks
)

# Print results
print(f"Search query: '{search_query}'\n")
print("Top similar documents:")
for i, doc in enumerate(similar_docs):
    print(f"\nDocument {i+1}:")
    print(f"Source: {doc.metadata.get('source', 'Unknown')}, Page: {doc.metadata.get('page', 'Unknown')}")
    print("-" * 80)
    print(doc.page_content)
    print("-" * 80)